# AE4 S30 — Proyecto Integrador: Métricas y Visualización de Datos en Streamlit

**Asignatura:** Proyecto Integrado III - Analítica de Datos (S30) — IU Digital de Antioquia  
**Estudiante:** Raul Santiago Morales Henao (`shenao356`) — **Grupo 6**  
**Docente:** Andrés Felipe Callejas Jaramillo  
**Dataset Base:** [`MongoDB/whatscooking.restaurants`](https://huggingface.co/datasets/MongoDB/whatscooking.restaurants) (`split="train"`) enriquecido con el Censo Oficial de NYC (`25.159` restaurantes × `27` variables, `0` nulos).  
**Aplicación Web Streamlit:** `src/clases_po/app.py` y `AE4/app.py`

---

## 1. Continuidad Metodológica (EA1 → AE2 → AE3 → AE4) y Diseño de Métricas

En esta cuarta entrega (**AE4 - Etapa 3: Métricas y Visualización de Datos**) retomamos el conjunto de datos depurado y enriquecido en la **AE3** para construir el sistema formal de métricas, las visualizaciones interactivas en **Plotly** y la aplicación web en **Streamlit**:

| # | Pregunta de Negocio | Métrica, Cálculo y Unidad | Gráfico Seleccionado | Por qué ese Gráfico | Decisión que Apoya |
|---|---|---|---|---|---|
| **PN1** | ¿Cómo se distribuyen los restaurantes y reseñas entre los 5 distritos de NYC y cuál es su densidad y reputación? | • Oferta: $N_i$ y $\%_i$ (locales y %)<br>• Densidad física: $N_i / \text{km}^2$ (locales/km²)<br>• Densidad censal: $(N_i / \text{Pob}_i)\times 10^4$ (locales/10k hab.) | Barras horizontales ordenadas con color fijo por distrito y barras agrupadas de densidad | Permite comparar magnitudes territoriales dispares y leer etiquetas sin rotar el texto | Descentralizar aperturas hacia distritos de alta tracción y menor saturación por $\text{km}^2$ (Brooklyn y Queens) |
| **PN2** | ¿Cuáles son las especialidades y las 8 macro-familias culinarias predominantes y cómo se comparan en tracción? | • Oferta por familia/cocina (locales y %)<br>• Promedio de reseñas ($\bar{v}$, opiniones/local)<br>• Popularidad ($\text{Calif} \times \text{Reseñas}$, pts) | Barras horizontales de las 8 familias, barras de tracción con resaltado de líderes y Top 15 cocinas | Contrasta volumen de competencia vs. demanda media por local en 84 categorías | Priorizar conceptos con identidad cultural diferenciada (Cocina Latina, Española, Cafeterías) frente a *American* genérico |
| **PN3** | ¿Cómo se segmenta la oferta por categoría de precio (declarada e imputada) y cómo varía su satisfacción y tracción? | • Participación por rango (`$` a `$$$$` y No Especif., %)<br>• Media y mediana de reseñas (opiniones/local)<br>• Reputación Bayesiana ($\text{IRP}$, estrellas ★) | Columnas agrupadas en orden natural (`$ → $$$$`), curva de mediana y contraste Simple vs. Bayesiana | Respeta el orden ordinal del precio y demuestra cómo el suavizado bayesiano corrige sesgos | Posicionar el restaurante en el segmento Moderado (`$$`) o Costoso (`$$$`), donde la tracción se duplica o triplica |
| **PN4** | ¿Qué proporción ofrece servicios de conveniencia y cómo incide el nivel de equipamiento en su popularidad? | • Adopción por servicio (% con `True`)<br>• Nivel de equipamiento (`1 a 4 servicios`): media/mediana de reseñas, popularidad (pts) y % $\ge 4,0\star$ | Barras horizontales de adopción individual y columnas agrupadas por Nivel de Equipamiento (`Básico`, `Intermedio`, `Completo`) | Distingue servicios estándar (TakeOut, Terraza) de diferenciadores (Happy Hour, Mascotas) | Diseñar el local bajo formato Multipropósito (3–4 servicios activos) para multiplicar ocasiones de consumo |
| **PN5** | ¿Cómo se cruzan el distrito y la categoría de precio en densidad de oferta, reseñas y reputación? | • Matriz bivariada Distrito × Precio (% intra-distrito, reseñas/local, popularidad en pts)<br>• Coordenadas (`Latitud`, `Longitud`) | Mapas de calor (*Heatmaps*) de 20 celdas (5 distritos × 4 precios) y mapa geográfico interactivo (*Scattermap*) | Revela en dos dimensiones simultáneas los nichos de alta demanda y baja saturación geográfica | Abrir concepto `$$` o `$$$` en corredores gastronómicos de Brooklyn o Queens con alta tracción por local |

## 2. Preparar el entorno e importar las clases del proyecto (`AnalisisRestaurantesNYC` y `DashboardStreamlit`)

El notebook localiza automáticamente la raíz del repositorio (`ROOT`) para importar las clases desde `src/clases_po` (y si se ejecuta en un entorno en la nube sin el repositorio clonado, descarga automáticamente los módulos y el dataset preparado desde la rama `main` de GitHub).

In [1]:
import sys
import urllib.request
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

REPO_RAW = "https://raw.githubusercontent.com/shenao356/PIntegrado_AE_Grupo-6/main"

if not (ROOT / "src" / "clases_po" / "dashboard_streamlit.py").exists():
    ROOT = Path.cwd().resolve()
    (ROOT / "src" / "clases_po").mkdir(parents=True, exist_ok=True)
    (ROOT / "AE4" / "datos").mkdir(parents=True, exist_ok=True)
    for mod in ["__init__.py", "analisis.py", "dashboard_streamlit.py", "reportes.py"]:
        urllib.request.urlretrieve(f"{REPO_RAW}/src/clases_po/{mod}", ROOT / "src" / "clases_po" / mod)
    urllib.request.urlretrieve(
        f"{REPO_RAW}/AE3/datos/whatscooking_restaurants_preparado.csv",
        ROOT / "AE4" / "datos" / "whatscooking_restaurants_preparado.csv",
    )

sys.path.insert(0, str(ROOT / "src"))

from clases_po import AnalisisRestaurantesNYC, DashboardStreamlit, Reportes

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
print("Raíz del proyecto configurada correctamente.")


Raíz del proyecto configurada correctamente.


## 3. Cargar el dataset preparado de la AE3 y sincronizar la copia de la aplicación (`AE4/datos/`)

La aplicación de Streamlit lee el dataset preparado y enriquecido (`25.159` restaurantes y `27` columnas con `0` valores nulos). El método `DashboardStreamlit.exportar_datos()` asegura que `AE4/datos/whatscooking_restaurants_preparado.csv` esté sincronizado con la salida oficial de la `AE3`.

In [2]:
ORIGEN_AE3 = ROOT / "AE3" / "datos" / "whatscooking_restaurants_preparado.csv"
DATOS_APP = ROOT / "AE4" / "datos" / "whatscooking_restaurants_preparado.csv"

if ORIGEN_AE3.exists():
    DashboardStreamlit.exportar_datos(ORIGEN_AE3, DATOS_APP)

dash = DashboardStreamlit.desde_csv(DATOS_APP)
print("Alcance:", dash.alcance_texto())
print(f"Dimensiones verificadas: {dash.R.shape[0]:,} filas × {dash.R.shape[1]} columnas | Total nulos: {int(dash.R.isna().sum().sum())}")


25,159 filas × 27 columnas → C:\Users\shena\OneDrive\Desktop\EA1 VERSION 2\AE4\datos\whatscooking_restaurants_preparado.csv (8.33 MB)


Alcance: Corte transversal NYC (Censo 2024 / Hugging Face) · 25.159 restaurantes (100,0 % del total) · 5 distrito(s) · 84 especialidades culinarias
Dimensiones verificadas: 25,159 filas × 27 columnas | Total nulos: 0


## 4. Indicadores Clave (KPIs) y Diccionario Formal de Métricas

Tanto el notebook como el encabezado de la aplicación en Streamlit presentan los mismos **6 KPIs ejecutivos** y utilizan las **9 métricas formales (`M1` a `M9`)** documentadas a continuación:

In [3]:
display(dash.kpis_tabla())
display(dash.A.tabla_definicion_metricas())


,Indicador (KPI),Valor Calculado
0,Restaurantes analizados (locales),25.159
1,% de la base total de NYC,"100,0 %"
2,Reseñas acumuladas (opiniones),2.415.896
3,Promedio de reseñas por local,"96,03 reseñas"
4,Calificación promedio simple,"3,53 ★"
5,Índice de reputación bayesiana,"3,56 ★"
6,Puntaje de popularidad promedio,"358,78 pts"
7,"Tasa de alta reputación (≥ 4,0 ★)","43,64 %"
8,Densidad comercial superficial,"32,20 locales/km²"
9,Densidad comercial demográfica,"28,58 locales/10k hab."


,Código,Métrica / Indicador,Fórmula de Cálculo,Unidad de Medida,Preguntas que Responde,Propósito Analítico
0,M1,Oferta y Participación de Restaurantes,N_i = count(ID_Restaurante) | %_i = (N_i / N...,Establecimientos (locales) y porcentaje (%),"PN1, PN2, PN3, PN4, PN5",Cuantifica el tamaño absoluto y peso relativo ...
1,M2,Densidad Comercial Superficial,Densidad_km2 = N_distrito / Superficie_Km2,Restaurantes por km² (locales/km²),PN1,Mide la saturación territorial física de estab...
2,M3,Densidad Comercial Demográfica,Densidad_10k = (N_distrito / Poblacion_Distrit...,Restaurantes por cada 10.000 habitantes (local...,PN1,Contextualiza la oferta gastronómica frente a ...
3,M4,Volumen y Promedio de Reseñas (Tracción),Suma = sum(Total_Resenas) | Media = mean(Tot...,Reseñas (opiniones de clientes),"PN1, PN2, PN3, PN4, PN5",Aproxima el flujo de interacción y visibilidad...
4,M5,Calificación Promedio Simple,R_bar = mean(Calificacion),"Estrellas (escala de 1,0 a 5,0 ★)","PN1, PN2, PN3, PN4, PN5",Mide la satisfacción directa declarada por los...
5,M6,Índice de Reputación Ponderada (Bayesiana),IRP = (v / (v + m)) * R + (m / (v + m)) * C ...,"Estrellas ponderadas (escala de 1,0 a 5,0 ★)","PN1, PN2, PN3, PN4, PN5",Corrige el sesgo de muestras pequeñas estabili...
6,M7,Puntaje de Popularidad (Tracción Comercial),Popularidad = Calificacion * Total_Resenas,Puntos de popularidad ponderada (pts),"PN1, PN2, PN3, PN4, PN5",Combina satisfacción y volumen de reseñas para...
7,M8,"Tasa de Alta Reputación (≥ 4,0 ★)",% Alta Rep. = (count(Calificacion >= 4.0) / N_...,Porcentaje de locales sobresalientes/excelente...,"PN1, PN2, PN3, PN4, PN5",Mide la proporción de restaurantes que alcanza...
8,M9,Equipamiento de Servicios de Conveniencia,Total_Servicios = Para_Llevar + Mesas_Exterior...,Cantidad de servicios activos (1 a 4 servicios...,PN4,Evalúa la dotación operativa del local (Básico...


### 4.1 Estadísticos descriptivos de las variables cuantitativas clave
Se verifica que las medidas de tendencia central, dispersión y posición coincidan exactamente con el reporte de la AE3:

In [4]:
display(dash.A.medidas_variables())


,Calificacion,Indice_Reputacion_Ponderada,Total_Resenas,Puntaje_Popularidad,Rango_Precio,Total_Servicios,Cantidad_Platos_Menu
n (datos),25159.000,25159.000,25159.000,25159.000,25159.000,25159.000,25159.000
Suma,88822.500,89672.458,2415896.000,9026636.000,40978.000,56841.000,171173.000
Media,3.530,3.564,96.025,358.784,1.629,2.259,6.804
Mediana,3.500,3.519,39.000,135.000,2.000,2.000,12.000
Moda,4.000,3.508,5.000,36.000,2.000,2.000,12.000
Mínimo,1.000,1.558,5.000,5.000,1.000,1.000,0.000
Máximo,5.000,4.916,9185.000,32485.500,4.000,4.000,14.000
Rango,4.000,3.358,9180.000,32480.500,3.000,3.000,14.000
Desviación estándar,0.792,0.383,193.604,771.659,0.580,0.886,6.036
Coef. de variación,0.224,0.107,2.016,2.151,0.356,0.392,0.887


---
## 5. Respuestas Dinámicas a las Cinco Preguntas de Negocio (`dash.respuestas()`)

El método `dash.respuestas()` redacta las respuestas utilizando directamente las cifras calculadas por `AnalisisRestaurantesNYC`, especificando la **métrica y unidad**, el **gráfico seleccionado**, la **decisión que apoya** y las **limitaciones metodológicas** (distinguiendo nivel de precios y volumen de reseñas frente a ingresos o rentabilidad neta):

In [5]:
for r in dash.respuestas():
    display(Markdown(
        f"### {r['id']} · {r['titulo']}\n"
        f"**Pregunta:** *{r['pregunta']}*\n\n"
        f"- **Métrica, cálculo y unidad:** {r['metrica']}\n"
        f"- **Respuesta con cifras del proyecto:** {r['respuesta']}\n"
        f"- **Gráfico seleccionado:** {r['visual']}\n"
        f"- **Decisión que apoya:** {r['decision']}\n"
        f"- **Limitaciones (ingresos vs. rentabilidad y no causalidad):** {r['limitacion']}"
    ))


### PN1 · Concentración geográfica y densidad demográfica por distrito
**Pregunta:** *¿Cómo se distribuye la oferta de restaurantes y el flujo de reseñas entre los cinco distritos de Nueva York, y cuál es su densidad comercial (por km² y por 10.000 habitantes) y reputación promedio?*

- **Métrica, cálculo y unidad:** Oferta (% locales = N_i / N_total × 100, en % y establecimientos), Densidad superficial (N_i / Superficie_Km2, en locales/km²), Densidad demográfica (N_i / Poblacion_Distrito × 10.000, en locales/10k hab.) y Reputación ponderada bayesiana (en estrellas ★ de 1,0 a 5,0).
- **Respuesta con cifras del proyecto:** Manhattan concentra la mayor oferta con 10.213 restaurantes (40,59 %) y 982.074 reseñas (40,65 %), alcanzando una saturación de 172,81 locales/km² y 60,28 locales/10k hab. (frente a Staten Island con 963 locales y 6,36 locales/km²). Sin embargo, la tracción media por local es muy pareja (~95–97 reseñas), destacando Queens en popularidad promedio (363,67 pts) y Staten Island en alta reputación (46,11 %).
- **Gráfico seleccionado:** Gráfico de barras horizontales ordenadas con color fijo de identidad por distrito y gráfico comparativo de densidad física (locales/km²) frente a densidad demográfica (locales/10k hab.).
- **Decisión que apoya:** Descentralizar la apertura hacia distritos con alta demanda por local pero menor saturación por km² (como Brooklyn y Queens, con 5 a 8 veces menos densidad superficial que Manhattan), donde el flujo de reseñas por establecimiento iguala o supera al centro financiero.
- **Limitaciones (ingresos vs. rentabilidad y no causalidad):** La densidad mide establecimientos registrados frente a población residente censal (U.S. Census Bureau, 2024); no captura la población flotante diaria (turistas y oficinas en Manhattan) ni mide ventas en dólares o costo del metro cuadrado de arriendo.

### PN2 · Especialidades y macro-familias culinarias predominantes
**Pregunta:** *¿Cuáles son las especialidades culinarias y las 8 macro-familias gastronómicas predominantes en Nueva York, y cómo se comparan en calificación, promedio de reseñas y popularidad?*

- **Métrica, cálculo y unidad:** Participación de oferta por cocina/familia (% de locales), Promedio de reseñas por local (mean(Total_Resenas), en opiniones/local) y Puntaje de Popularidad (Calificacion × Total_Resenas, en puntos).
- **Respuesta con cifras del proyecto:** En volumen de oferta lidera la familia 'Norteamericana' con 7.231 locales (28,74 %), encabezada por la cocina 'American' (6.173 locales; 24,54 %). No obstante, en tracción promedio por establecimiento lideran 'Otras Cocinas del Mundo' (105,33 reseñas y 397,69 pts) y dentro del Top 15 destaca 'Spanish' (111,27 reseñas y 413,22 pts).
- **Gráfico seleccionado:** Barras horizontales de volumen por las 8 macro-familias, gráfico comparativo de tracción promedio (reseñas y popularidad) y ranking Top 15 de especialidades individuales.
- **Decisión que apoya:** Evitar competir con propuestas genéricas en la categoría más saturada ('American' tradicional) y apostar por conceptos gastronómicos con identidad cultural diferenciada (cocina hispana/latina, mediterránea/española o cafeterías de especialidad), que generan mayor volumen de reseñas por local con menor saturación.
- **Limitaciones (ingresos vs. rentabilidad y no causalidad):** Un mayor volumen de reseñas y popularidad refleja visibilidad e interacción de los usuarios en la plataforma, pero no equivale directamente a mayores ingresos netos ni a mayor margen de rentabilidad (que dependen del costo de insumos y rotación de mesas).

### PN3 · Segmentos de rango de precio, satisfacción y popularidad
**Pregunta:** *¿Cómo se segmenta la oferta gastronómica según las categorías de rango de precio (declarada e imputada) y qué relación guarda el nivel de precios con la satisfacción y el volumen de reseñas?*

- **Métrica, cálculo y unidad:** Distribución porcentual por nivel de precio ($ a $$$$ y No Especificado, en %), Promedio y mediana de reseñas (opiniones/local) e Índice de Reputación Ponderada Bayesiana (en ★).
- **Respuesta con cifras del proyecto:** Los segmentos Económico ($) y Moderado ($$) concentran el 95,89 % de la oferta total. Al subir desde '1 - Económico ($)' hasta '3 - Costoso ($$$)', el promedio de reseñas pasa de 62,13 a 179,53 opiniones/local y la popularidad sube de 230,13 a 688,29 pts. El segmento sin precio declarado (2.394 locales; 9,52 %) presenta pocas reseñas (23,65 en promedio) y una calificación simple inflada (3,93 ★), que el suavizado bayesiano corrige a 3,67 ★.
- **Gráfico seleccionado:** Columnas agrupadas en orden ordinal natural ($ → $$$$) comparando precio declarado vs. imputado, curva/barras de gradiente de tracción y contraste entre calificación simple y reputación bayesiana.
- **Decisión que apoya:** Posicionar el modelo de negocio en el segmento Moderado ($$) o Costoso de entrada ($$$), donde se concentra la mayor tracción de reseñas y mejor reputación ponderada, evitando la guerra de precios del segmento Económico ($).
- **Limitaciones (ingresos vs. rentabilidad y no causalidad):** La categoría de precio ($ a $$$$) indica el nivel tarifario percibido de la carta y NO los ingresos totales ni la rentabilidad financiera del restaurante. Además, el 9,52 % sin precio fue imputado con la mediana de (Tipo_Cocina, Barrio) para análisis multivariado.

### PN4 · Disponibilidad de servicios de conveniencia y valor agregado
**Pregunta:** *¿Qué proporción de restaurantes ofrece servicios de conveniencia (para llevar, mesas exteriores, hora feliz, mascotas) y cómo varía la popularidad y reputación según el nivel de equipamiento?*

- **Métrica, cálculo y unidad:** Tasa de adopción por servicio (% de locales con True) y desempeño por Nivel de Equipamiento (Total_Servicios de 1 a 4: promedio/mediana de reseñas, popularidad en pts y % de alta reputación).
- **Respuesta con cifras del proyecto:** Comida para llevar (100,0 %) y mesas exteriores (80,0 %) son estándares del mercado, mientras que hora feliz (23,5 %) y admisión de mascotas (22,4 %) actúan como diferenciadores. Los locales con equipamiento '3. Completo / Multipropósito (3-4 servicios)' (31,16 % del total) alcanzan 189,80 reseñas promedio (6,7 veces más que el nivel Básico con 28,27), calificación de 3,76 ★ y 55,61 % de alta reputación.
- **Gráfico seleccionado:** Barras horizontales de adopción de los 4 servicios individuales y gráficos de columnas por Nivel de Equipamiento (1. Básico, 2. Intermedio, 3. Completo/Multipropósito).
- **Decisión que apoya:** Diseñar el establecimiento bajo un formato multipropósito (3 a 4 servicios activos: para llevar + terraza exterior + franja de happy hour + política pet-friendly en terraza) para maximizar ocasiones de consumo y visibilidad digital.
- **Limitaciones (ingresos vs. rentabilidad y no causalidad):** La fuerte diferencia en reseñas y calificación según el número de servicios es una asociación observacional y NO demuestra causalidad automática: habilitar terraza o happy hour implica costos de licencias, personal e infraestructura que deben evaluarse financieramente.

### PN5 · Cruce estratégico Distrito × Categoría de Precio y Mapa de NYC
**Pregunta:** *¿Cómo se cruzan el distrito geográfico y la categoría de precio en densidad de oferta, reseñas y reputación para identificar zonas saturadas frente a nichos de oportunidad?*

- **Métrica, cálculo y unidad:** Matriz bivariada Distrito × Precio: Participación intra-distrito (% locales), Promedio de reseñas (opiniones/local), Popularidad promedio (pts) y Georreferenciación (Latitud, Longitud en NYC).
- **Respuesta con cifras del proyecto:** En todos los distritos la estructura interna se concentra en $ (~41–43 %) y $$ (~52–55 %), mientras que $$$ (~3,4–3,7 %) y $$$$ (~0,4–0,7 %) son nichos selectos. En el segmento Costoso ($$$), Brooklyn lidera la tracción con 208,69 reseñas/local y 799,28 pts de popularidad en solo 213 locales. Asimismo, Queens y Staten Island en el segmento Moderado ($$) superan las 113–120 reseñas promedio con calificaciones de 3,61 ★ a 3,64 ★.
- **Gráfico seleccionado:** Mapas de calor (Heatmaps) de oferta y tracción en las 20 celdas (5 distritos × 4 niveles de precio) y mapa geográfico interactivo (Scattermap) de Nueva York.
- **Decisión que apoya:** Priorizar la apertura de un concepto Moderado-Alto ($$ o $$$) en corredores gastronómicos de Brooklyn o Queens, aprovechando su alta tracción por restaurante y menor saturación competidora frente a Manhattan.
- **Limitaciones (ingresos vs. rentabilidad y no causalidad):** El análisis espacial identifica vacíos de oferta y alta tracción a nivel de distrito y código postal, pero la selección final del local requiere un estudio micro-inmobiliario de tráfico peatonal, zonificación y flujo de caja proyectado.

---
## 6. Visualizaciones Interactivas y Tablas de Resultados por Pregunta de Negocio

Cada método `figura_*()` de `DashboardStreamlit` devuelve un objeto `plotly.graph_objects.Figure` que es exactamente el mismo que se renderiza en las pestañas de la aplicación web de Streamlit.

### 6.1 Pregunta de Negocio 1 (PN1): Distribución geográfica y densidad demográfica por distrito

In [6]:
display(dash.A.pn1_distribucion_distritos(incluir_total=True))
fig_pn1_a = dash.figura_pn1_oferta_distritos()
fig_pn1_b = dash.figura_pn1_densidad_territorial()
fig_pn1_c = dash.figura_pn1_traccion_distritos()
print("Figuras PN1 generadas:", [fig_pn1_a.layout.title.text.split('<br>')[0], fig_pn1_b.layout.title.text.split('<br>')[0], fig_pn1_c.layout.title.text.split('<br>')[0]])


,Barrio,Restaurantes,Participacion_Locales_Pct,Resenas_Acumuladas,Participacion_Resenas_Pct,Promedio_Resenas,Calificacion_Promedio,Reputacion_Ponderada,Popularidad_Promedio,Locales_por_Km2,Locales_por_10k_Hab,Pct_Alta_Reputacion,Poblacion_Distrito,Superficie_Km2,Vocacion_Comercial
0,Manhattan,10213,40.59,982074,40.65,96.16,3.52,3.56,360.33,172.81,60.28,43.06,1694251,59.1,"Centro financiero, turístico y corporativo"
1,Brooklyn,6046,24.03,574809,23.79,95.07,3.54,3.57,353.97,33.59,22.10,44.08,2736074,180.0,"Residencial denso, cultural y gastronómico eme..."
2,Queens,5615,22.32,546034,22.60,97.25,3.54,3.57,363.67,19.94,23.34,43.85,2405464,281.6,"Multicultural, residencial y aeroportuario"
3,Bronx,2322,9.23,220785,9.14,95.08,3.52,3.55,352.28,21.26,15.77,43.50,1472654,109.2,Residencial urbano y comercio barrial
4,Staten Island,963,3.83,92194,3.82,95.74,3.55,3.58,359.81,6.36,19.43,46.11,495747,151.5,Suburbano y baja densidad comercial
5,TOTAL / GLOBAL NYC,25159,100.00,2415896,100.00,96.03,3.53,3.56,358.78,32.20,28.58,43.64,8804190,781.4,Consolidado distritos seleccionados


Figuras PN1 generadas: ['<b>PN1 · Oferta de restaurantes y participación (%) por distrito</b>', '<b>PN1 · Densidad física (locales/km²) vs. demográfica (locales/10k hab.)</b>', '<b>PN1 · Popularidad promedio y % de alta reputación por distrito</b>']


### 6.2 Pregunta de Negocio 2 (PN2): Especialidades culinarias y las 8 macro-familias gastronómicas

In [7]:
display(dash.A.pn2_familias_culinarias())
display(dash.A.pn2_especialidades_culinarias(top_n=15))
fig_pn2_a = dash.figura_pn2_familias_oferta()
fig_pn2_b = dash.figura_pn2_familias_traccion()
fig_pn2_c = dash.figura_pn2_top_cocinas(n=15, metrica="Popularidad_Promedio")
print("Figuras PN2 generadas:", [fig_pn2_a.layout.title.text.split('<br>')[0], fig_pn2_b.layout.title.text.split('<br>')[0], fig_pn2_c.layout.title.text.split('<br>')[0]])


,Familia_Culinaria,Restaurantes,Resenas_Acumuladas,Promedio_Resenas,Calificacion_Promedio,Reputacion_Ponderada,Popularidad_Promedio,Precio_Promedio,Pct_Alta_Reputacion,Participacion_Locales_Pct,Participacion_Resenas_Pct
0,Norteamericana,7231,691409,95.62,3.53,3.56,355.88,1.63,43.76,28.74,28.62
1,Europea y Mediterránea,4639,439440,94.73,3.54,3.57,355.08,1.63,44.60,18.44,18.19
2,Asiática,4584,434144,94.71,3.53,3.56,352.09,1.64,43.00,18.22,17.97
3,"Cafetería, Panadería y Postres",2918,274162,93.96,3.55,3.57,350.15,1.62,44.04,11.60,11.35
4,Latinoamericana y Caribeña,2512,259147,103.16,3.52,3.56,390.50,1.59,43.07,9.98,10.73
5,Otras Cocinas del Mundo,1641,172850,105.33,3.54,3.57,397.69,1.63,44.12,6.52,7.15
6,"Sandwiches, Deli y Casual",1137,98786,86.88,3.50,3.55,321.95,1.61,42.04,4.52,4.09
7,"Saludable, Vegetariana y Bebidas",497,45958,92.47,3.51,3.56,343.49,1.65,41.45,1.98,1.90


,Tipo_Cocina,Familia_Culinaria,Restaurantes,Resenas_Acumuladas,Promedio_Resenas,Calificacion_Promedio,Reputacion_Ponderada,Popularidad_Promedio,Precio_Promedio,Pct_Alta_Reputacion,Participacion_Locales_Pct
0,American,Norteamericana,6173,579024,93.80,3.52,3.56,348.66,1.64,43.27,24.54
1,Chinese,Asiática,2415,223766,92.66,3.53,3.56,343.00,1.64,43.27,9.60
2,Café/Coffee/Tea,"Cafetería, Panadería y Postres",1212,119387,98.50,3.56,3.58,370.09,1.64,44.64,4.82
3,Pizza,Europea y Mediterránea,1162,106935,92.03,3.53,3.57,343.70,1.64,45.87,4.62
4,Italian,Europea y Mediterránea,1069,103392,96.72,3.53,3.56,366.07,1.66,43.59,4.25
5,Other,Otras Cocinas del Mundo,856,88328,103.19,3.51,3.57,390.20,1.63,41.24,3.40
6,Latin American,Latinoamericana y Caribeña,850,89109,104.83,3.52,3.56,397.20,1.62,43.41,3.38
7,Japanese,Asiática,758,68043,89.77,3.51,3.54,331.25,1.64,40.90,3.01
8,Mexican,Latinoamericana y Caribeña,753,74505,98.94,3.51,3.55,376.95,1.58,44.36,2.99
9,Bakery,"Cafetería, Panadería y Postres",689,53921,78.26,3.51,3.55,286.00,1.61,41.94,2.74


Figuras PN2 generadas: ['<b>PN2 · Oferta de restaurantes por macro-familia gastronómica (8 familias)</b>', '<b>PN2 · Tracción promedio por macro-familia (naranja = líderes en demanda)</b>', '<b>PN2 · Top 15 especialidades culinarias ordenadas por Popularidad Promedio</b>']


### 6.3 Pregunta de Negocio 3 (PN3): Segmentos de rango de precio (declarado vs. imputado), satisfacción y tracción

In [8]:
display(dash.A.pn3_segmentos_precio_declarado())
display(dash.A.pn3_segmentos_precio_imputado())
fig_pn3_a = dash.figura_pn3_declarado_vs_imputado()
fig_pn3_b = dash.figura_pn3_gradiente_traccion()
fig_pn3_c = dash.figura_pn3_calificacion_vs_bayesiana()
print("Figuras PN3 generadas:", [fig_pn3_a.layout.title.text.split('<br>')[0], fig_pn3_b.layout.title.text.split('<br>')[0], fig_pn3_c.layout.title.text.split('<br>')[0]])


,Categoria_Precio_Declarada,Restaurantes,Resenas_Acumuladas,Promedio_Resenas,Mediana_Resenas,Calificacion_Promedio,Reputacion_Ponderada,Popularidad_Promedio,Pct_Alta_Reputacion,Participacion_Locales_Pct,Participacion_Resenas_Pct
0,1 - Económico ($),10275,647507,63.02,31.0,3.42,3.53,233.31,41.07,40.84,26.80
1,2 - Moderado ($$),11459,1528375,133.38,63.0,3.54,3.57,497.23,41.27,45.55,63.26
2,3 - Costoso ($$$),899,161395,179.53,81.0,3.57,3.61,688.29,45.49,3.57,6.68
3,4 - Gourmet / Exclusivo ($$$$),132,22012,166.76,35.5,3.53,3.64,650.47,49.24,0.52,0.91
4,No Especificado,2394,56607,23.65,14.0,3.93,3.67,94.78,65.00,9.52,2.34


,Rango_Precio,Categoria_Precio,Restaurantes,Resenas_Acumuladas,Promedio_Resenas,Mediana_Resenas,Calificacion_Promedio,Reputacion_Ponderada,Popularidad_Promedio,Pct_Alta_Reputacion,Participacion_Locales_Pct,Participacion_Resenas_Pct
0,1,1 - Económico ($),10505,652658,62.13,30.0,3.43,3.53,230.13,41.59,41.75,27.02
1,2,2 - Moderado ($$),13622,1579798,115.97,49.0,3.60,3.59,433.44,45.04,54.14,65.39
2,3,3 - Costoso ($$$),899,161395,179.53,81.0,3.57,3.61,688.29,45.49,3.57,6.68
3,4,4 - Gourmet / Exclusivo ($$$$),133,22045,165.75,35.0,3.53,3.64,646.58,49.62,0.53,0.91


Figuras PN3 generadas: ['<b>PN3 · Oferta por categoría de precio: declarada vs. imputada</b>', '<b>PN3 · Tracción comercial (media y mediana de reseñas) por nivel de precio</b>', "<b>PN3 · Calificación simple vs. reputación bayesiana (corrección en 'Sin Precio Declarado')</b>"]


### 6.4 Pregunta de Negocio 4 (PN4): Disponibilidad de servicios de conveniencia y nivel de equipamiento

In [9]:
display(dash.A.pn4_adopcion_individual())
display(dash.A.pn4_servicios_conveniencia())
fig_pn4_a = dash.figura_pn4_adopcion_individual()
fig_pn4_b = dash.figura_pn4_impacto_equipamiento()
fig_pn4_c = dash.figura_pn4_reputacion_equipamiento()
print("Figuras PN4 generadas:", [fig_pn4_a.layout.title.text.split('<br>')[0], fig_pn4_b.layout.title.text.split('<br>')[0], fig_pn4_c.layout.title.text.split('<br>')[0]])


,Columna,Servicio,Restaurantes_Activos,Adopcion_Pct,Restaurantes_No_Activos,Promedio_Resenas_Si,Promedio_Resenas_No,Popularidad_Si,Calificacion_Si
0,Para_Llevar,Comida para llevar (TakeOut),25159,100.00,0,96.03,0.00,358.78,3.53
1,Mesas_Exteriores,Mesas al aire libre (OutdoorSeating),20127,80.00,5032,111.54,33.97,416.59,3.52
2,Hora_Feliz,Hora feliz / promociones (HappyHour),5917,23.52,19242,201.38,63.63,773.12,3.74
3,Permite_Mascotas,Admisión de mascotas (DogsAllowed),5638,22.41,19521,202.90,65.16,794.74,3.88


,Nivel_Servicios,Restaurantes,Resenas_Acumuladas,Promedio_Resenas,Mediana_Resenas,Calificacion_Promedio,Reputacion_Ponderada,Popularidad_Promedio,Pct_Alta_Reputacion,Pct_Para_Llevar,Pct_Mesas_Exteriores,Pct_Hora_Feliz,Pct_Permite_Mascotas,Participacion_Locales_Pct,Participacion_Resenas_Pct
0,1. Básico (1 servicio),4388,124068,28.27,15.0,3.50,3.54,102.87,45.40,100.0,0.00,0.00,0.00,17.44,5.14
1,2. Intermedio (2 servicios),12932,803977,62.17,33.0,3.40,3.49,221.26,35.79,100.0,95.61,0.94,3.46,51.40,33.28
2,3. Completo / Multipropósito (3-4 servicios),7839,1487851,189.80,105.0,3.76,3.69,728.92,55.61,100.0,99.03,73.94,66.22,31.16,61.59


Figuras PN4 generadas: ['<b>PN4 · Tasa de adopción de los 4 servicios de conveniencia</b>', '<b>PN4 · Volumen de reseñas (media y mediana) por Nivel de Equipamiento</b>', '<b>PN4 · Popularidad promedio, calificación y % de alta reputación por Nivel de Servicios</b>']


### 6.5 Pregunta de Negocio 5 (PN5): Matriz multivariada Distrito × Categoría de Precio y Mapa Geográfico de NYC

In [10]:
display(dash.A.pn5_matriz_distrito_precio())
fig_pn5_a = dash.figura_pn5_heatmap_oferta()
fig_pn5_b = dash.figura_pn5_heatmap_traccion(metrica="Promedio_Resenas")
fig_pn5_c = dash.figura_pn5_mapa_nyc(barrio="Brooklyn")
print("Figuras PN5 generadas:", [fig_pn5_a.layout.title.text.split('<br>')[0], fig_pn5_b.layout.title.text.split('<br>')[0], fig_pn5_c.layout.title.text.split('<br>')[0]])


,Barrio,Rango_Precio,Categoria_Precio,Restaurantes,Resenas_Acumuladas,Promedio_Resenas,Calificacion_Promedio,Reputacion_Ponderada,Popularidad_Promedio,Pct_Alta_Reputacion,Participacion_En_Distrito_Pct,Participacion_Global_Pct
0,Bronx,1,1 - Económico ($),998,57626,57.74,3.43,3.53,214.52,42.38,42.98,3.97
1,Bronx,2,2 - Moderado ($$),1234,150502,121.96,3.59,3.57,451.41,43.92,53.14,4.90
2,Bronx,3,3 - Costoso ($$$),78,10024,128.51,3.56,3.60,474.78,51.28,3.36,0.31
3,Bronx,4,4 - Gourmet / Exclusivo ($$$$),12,2633,219.42,3.75,3.75,819.25,41.67,0.52,0.05
4,Brooklyn,1,1 - Económico ($),2538,151077,59.53,3.46,3.54,221.21,43.30,41.98,10.09
5,Brooklyn,2,2 - Moderado ($$),3254,374017,114.94,3.60,3.58,426.76,44.59,53.82,12.93
6,Brooklyn,3,3 - Costoso ($$$),213,44450,208.69,3.60,3.62,799.28,46.01,3.52,0.85
7,Brooklyn,4,4 - Gourmet / Exclusivo ($$$$),41,5265,128.41,3.48,3.58,480.98,41.46,0.68,0.16
8,Manhattan,1,1 - Económico ($),4201,259836,61.85,3.41,3.52,228.20,40.16,41.13,16.70
9,Manhattan,2,2 - Moderado ($$),5597,650560,116.23,3.60,3.59,436.31,45.06,54.80,22.25


Figuras PN5 generadas: ['<b>PN5 · Mapa de calor de oferta: concentración por Distrito y Nivel de Precio</b>', '<b>PN5 · Mapa de calor de tracción: promedio de reseñas por local (Distrito × Precio)</b>', '<b>PN5 · Mapa geográfico de restaurantes en NYC (Brooklyn) — color = precio, tamaño = reseñas</b>']


---
## 7. Prueba del Motor de Filtros Interactivos (`dash.filtrar()`)

El método `filtrar()` crea una nueva vista del dashboard cuando el usuario interactúa con la barra lateral de Streamlit, conservando la paleta de colores de identidad por distrito. A continuación se simula la selección estratégica recomendada (**Brooklyn y Queens**, segmentos **Moderado `$$` y Costoso `$$$`** y **Equipamiento Completo de 3–4 servicios**):

In [11]:
nicho_estrategico = dash.filtrar(
    barrios=["Brooklyn", "Queens"],
    precios=["2 - Moderado ($$)", "3 - Costoso ($$$)"],
    servicios=["3. Completo / Multipropósito (3-4 servicios)"],
)
print("Alcance filtrado:", nicho_estrategico.alcance_texto())
display(nicho_estrategico.kpis_tabla())


Alcance filtrado: Corte transversal NYC (Censo 2024 / Hugging Face) · 2.627 restaurantes (10,4 % del total) · 2 distrito(s) · 70 especialidades culinarias


,Indicador (KPI),Valor Calculado
0,Restaurantes analizados (locales),2.627
1,% de la base total de NYC,"10,4 %"
2,Reseñas acumuladas (opiniones),543.529
3,Promedio de reseñas por local,"206,90 reseñas"
4,Calificación promedio simple,"3,74 ★"
5,Índice de reputación bayesiana,"3,68 ★"
6,Puntaje de popularidad promedio,"788,44 pts"
7,"Tasa de alta reputación (≥ 4,0 ★)","52,87 %"
8,Densidad comercial superficial,"5,69 locales/km²"
9,Densidad comercial demográfica,"5,11 locales/10k hab."


---
## 8. Verificación Automática de la Aplicación Streamlit sin Navegador (`AppTest`)

Siguiendo la metodología de la Semana 6, se utiliza `streamlit.testing.v1.AppTest` para ejecutar la aplicación completa (`src/clases_po/app.py` y `AE4/app.py`) en modo automatizado, comprobando que renderiza todas las métricas, pestañas y filtros con **cero excepciones**:

In [12]:
import logging
from streamlit.testing.v1 import AppTest

logging.getLogger("streamlit").setLevel(logging.ERROR)

at = AppTest.from_file(str(ROOT / "src" / "clases_po" / "app.py"), default_timeout=180).run()
print("Excepciones al iniciar la app:", len(at.exception))
print("KPIs en cabecera:", [(m.label, m.value) for m in at.metric])
print("Pestañas cargadas:", [t.label for t in at.tabs])
assert not at.exception, at.exception

# Prueba interactiva de filtro en la barra lateral (seleccionando únicamente Brooklyn y Queens)
at.sidebar.multiselect[0].set_value(["Brooklyn", "Queens"]).run()
print("Excepciones tras filtrar por Brooklyn y Queens:", len(at.exception))
print("KPIs recalculados con filtro:", [(m.label, m.value) for m in at.metric])
assert not at.exception, at.exception


2026-11-17 08:43:44.388 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Excepciones al iniciar la app: 0
KPIs en cabecera: [('Restaurantes Activos', '25.159'), ('Reseñas Acumuladas', '2.415.896'), ('Calificación Promedio', '3,53 ★'), ('Popularidad Promedio', '358,8 pts')]
Pestañas cargadas: ['🏙️ 1. Distritos (PN1)', '🍽️ 2. Cocinas (PN2)', '💲 3. Precios (PN3)', '🛎️ 4. Servicios (PN4)', '🗺️ 5. Matriz y Mapa (PN5)', '📌 6. Conclusiones y Datos']


Excepciones tras filtrar por Brooklyn y Queens: 0
KPIs recalculados con filtro: [('Restaurantes Activos', '11.661'), ('Reseñas Acumuladas', '1.120.843'), ('Calificación Promedio', '3,54 ★'), ('Popularidad Promedio', '358,6 pts')]
